# Cad West, Wales — Terrain Processing Pipeline Demo

Demonstrates the full LiteAeroSim terrain ingestion pipeline for a 100 km × 100 km grid
centred on **Cad West, Cambrian Mountains, Wales (52.84 °N, 3.72 °W)**:

1. Generate (or download) a DEM for the area
2. Triangulate to an LOD 3 TIN (≈ 300 m vertex spacing)
3. Simplify to LOD 4 (≈ 1 km) and LOD 5 (≈ 3 km)
4. Apply elevation-based colorization
5. Export to `.las_terrain` binary format
6. Visualise all three LODs with PyVista — interactive, with daylight sun illumination

**Real data download is optional.** The notebook runs fully offline using a synthetic
fractional-Brownian-motion heightfield that approximates Welsh upland terrain.

## 1 — Imports and path setup

In [ ]:
import sys
import math
import time
import tempfile
import dataclasses
from pathlib import Path

import numpy as np
from scipy.ndimage import gaussian_filter
import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_bounds

# Notebook lives in python/; tools are in python/tools/ and python/tools/terrain/.
_HERE = Path(".").resolve()
for _p in [str(_HERE / "tools"), str(_HERE / "tools" / "terrain")]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

from las_terrain import TerrainTileData, read_las_terrain, write_las_terrain
from triangulate import triangulate
from simplify import simplify
from verify import check, MeshQualityError
from export import export_las_terrain

print("Terrain tools loaded.")

## 2 — Location and grid parameters

Cad West sits in the Cambrian Mountains — the upland spine of mid-Wales. The area includes
rounded moorland plateaux, the Irfon/Wye valley system, and peaks rising to ~800 m ASL.

In [ ]:
# Cad West, Cambrian Mountains, Wales
CAD_WEST_LAT_DEG = 52.84   # degrees N
CAD_WEST_LON_DEG = -3.72   # degrees E (negative = West)

# 100 km x 100 km grid: +/-50 km from centroid
HALF_EXTENT_KM = 50.0
lat_half_deg = HALF_EXTENT_KM / 111.1
lon_half_deg = HALF_EXTENT_KM / (111.1 * math.cos(math.radians(CAD_WEST_LAT_DEG)))

# (lon_min, lat_min, lon_max, lat_max) — bbox_deg convention used by triangulate.py
BBOX_DEG = (
    CAD_WEST_LON_DEG - lon_half_deg,
    CAD_WEST_LAT_DEG - lat_half_deg,
    CAD_WEST_LON_DEG + lon_half_deg,
    CAD_WEST_LAT_DEG + lat_half_deg,
)
lon_min, lat_min, lon_max, lat_max = BBOX_DEG

print(f"Centroid : {CAD_WEST_LAT_DEG:.2f} N, {abs(CAD_WEST_LON_DEG):.2f} W")
print(f"Bbox     : [{lon_min:.3f}, {lat_min:.3f}] -> [{lon_max:.3f}, {lat_max:.3f}] deg")
print(f"Grid     : {2*HALF_EXTENT_KM:.0f} km x {2*HALF_EXTENT_KM:.0f} km")

## 3 — Synthetic DEM generation

A synthetic heightfield is generated using octave-summed Gaussian noise (fractional Brownian
motion approximation) with a large-scale plateau structure modelled on the Cambrian Mountains.
The result is saved to a temporary GeoTIFF so the standard `triangulate.py` pipeline can read it.

**Skip this cell and run §4 instead** if you have Copernicus DEM tiles already downloaded.

In [ ]:
def generate_welsh_terrain(ny: int, nx: int, seed: int = 42) -> np.ndarray:
    """Synthetic Welsh upland terrain via octave-summed Gaussian noise (fBm approximation).

    Returns a (ny, nx) float32 array with elevation in metres.
    """
    rng = np.random.default_rng(seed)
    h = np.zeros((ny, nx), dtype=np.float64)

    # Octave table: (blur sigma, amplitude) — coarse to fine.
    octaves = [
        (160, 1.00),  # large-scale ridges
        ( 70, 0.60),  # valley systems
        ( 30, 0.35),  # spurs and shoulders
        ( 12, 0.20),  # hillside texture
        (  5, 0.12),  # small knolls
        (  2, 0.07),  # micro-relief
        (  1, 0.04),  # high-frequency roughness
    ]
    for sigma, amp in octaves:
        raw = rng.standard_normal((ny, nx))
        smoothed = gaussian_filter(raw, sigma=sigma)
        s = smoothed.std()
        if s > 1e-10:
            smoothed /= s
        h += amp * smoothed

    # Large-scale topographic trend: two upland massifs (Cambrian + Brecon Beacons analogue)
    y_n = np.linspace(-1.0, 1.0, ny)
    x_n = np.linspace(-1.0, 1.0, nx)
    xx, yy = np.meshgrid(x_n, y_n)
    trend = (
        0.65 * np.exp(-((xx - 0.10) ** 2 / 1.0 + (yy + 0.05) ** 2 / 0.55))
        + 0.30 * np.exp(-((xx + 0.40) ** 2 / 0.45 + (yy - 0.35) ** 2 / 0.40))
    )
    h += trend

    # Scale to realistic Welsh range: 25 m (valley floors) to ~885 m (summit)
    h -= h.min()
    h = h / h.max() * 860.0 + 25.0
    return h.astype(np.float32)


# Resolution: ~3x finer than LOD 3 grid spacing (0.0027 deg) to ensure smooth sampling.
DEM_NY, DEM_NX = 500, 800

print(f"Generating {DEM_NX}x{DEM_NY} synthetic DEM ...", end=" ", flush=True)
t0 = time.time()
dem_array = generate_welsh_terrain(DEM_NY, DEM_NX)
print(f"done in {time.time()-t0:.1f}s")
print(f"  Elevation range: {dem_array.min():.0f} - {dem_array.max():.0f} m")

# Save to a temporary directory.
_TMP_DIR = Path(tempfile.mkdtemp(prefix="cad_west_"))
DEM_PATH = _TMP_DIR / "dem_synthetic.tif"

tfm = from_bounds(lon_min, lat_min, lon_max, lat_max, DEM_NX, DEM_NY)
with rasterio.open(
    DEM_PATH, "w", driver="GTiff",
    height=DEM_NY, width=DEM_NX, count=1,
    dtype="float32", crs=CRS.from_epsg(4326), transform=tfm,
) as dst:
    dst.write(dem_array, 1)

print(f"  Saved to: {DEM_PATH}")

## 4 — (Optional) Real data download

Set `USE_REAL_DATA = True` and configure credentials to download authentic terrain data.

**Copernicus DEM GLO-30** (30 m, already ellipsoidal — no geoid correction needed):
- Register at <https://dataspace.copernicus.eu/> and create an OAuth2 client
- Export env vars: `COPERNICUS_CLIENT_ID` and `COPERNICUS_CLIENT_SECRET`

**NASA EarthData** (SRTM / NASADEM, orthometric heights — geoid correction applied automatically):
- Register at <https://urs.earthdata.nasa.gov/> and configure `~/.netrc`

If credentials are absent the notebook continues with the synthetic DEM from §3.

In [ ]:
USE_REAL_DATA = False  # <- set True if credentials are configured

if USE_REAL_DATA:
    from download import download_dem, DownloadError
    from mosaic import mosaic_dem

    tiles_dir = _TMP_DIR / "dem_tiles"
    mosaic_path = _TMP_DIR / "dem_mosaic.tif"

    try:
        print("Downloading Copernicus DEM tiles ...")
        tile_paths = download_dem(BBOX_DEG, output_dir=tiles_dir, source="copernicus")
        print(f"  {len(tile_paths)} tile(s) downloaded")

        print("Mosaicking ...")
        mosaic_dem(tile_paths, mosaic_path)

        # Copernicus GLO-30 is already WGS84 ellipsoidal — no geoid step needed.
        DEM_PATH = mosaic_path
        print(f"  Using real DEM: {DEM_PATH}")

    except Exception as exc:
        print(f"Download failed ({exc}); continuing with synthetic DEM.")
else:
    print("USE_REAL_DATA = False — using synthetic DEM from §3.")

print(f"Active DEM: {DEM_PATH}")

## 5 — Triangulate at LOD 3 (≈ 300 m vertex spacing)

`triangulate.py` samples the DEM on a regular lon/lat grid at the LOD 3 spacing
(0.0027 ° ≈ 300 m) and runs `scipy.spatial.Delaunay` to build the base TIN.
Over the 100 km × 100 km bbox this produces roughly 330 × 550 = **180 k vertices**
and **360 k triangles**.

In [ ]:
print("Triangulating LOD 3 (0.0027 deg spacing) ...", end=" ", flush=True)
t0 = time.time()
tile_l3 = triangulate(DEM_PATH, BBOX_DEG, lod=3)
dt = time.time() - t0
nv3 = len(tile_l3.vertices)
nf3 = len(tile_l3.indices)
print(f"done in {dt:.1f}s")
print(f"  LOD 3: {nv3:,} vertices, {nf3:,} triangles")

## 6 — LOD simplification: LOD 3 → 4 → 5

`simplify.py` uses **pyfqmr QEM decimation** with `preserve_border=True` so that adjacent
tiles share crack-free edges. The target face count is scaled by the square of the spacing
ratio so that surface density is approximately consistent across LODs.

| LOD | Spacing | Target triangles |
|-----|---------|------------------|
| 3   | ≈ 300 m | ~360 k           |
| 4   | ≈ 1 km  | ~32 k            |
| 5   | ≈ 3 km  | ~3.5 k           |

In [ ]:
print("Simplifying LOD 3 -> LOD 4 ...", end=" ", flush=True)
t0 = time.time()
tile_l4 = simplify(tile_l3, target_lod=4)
nf4 = len(tile_l4.indices)
print(f"done in {time.time()-t0:.1f}s  ({nf4:,} triangles)")

print("Simplifying LOD 4 -> LOD 5 ...", end=" ", flush=True)
t0 = time.time()
tile_l5 = simplify(tile_l4, target_lod=5)
nf5 = len(tile_l5.indices)
print(f"done in {time.time()-t0:.1f}s  ({nf5:,} triangles)")

print()
print(f"Reduction ratios:  L3->L4 {nf3/nf4:.0f}x,  L4->L5 {nf4/nf5:.0f}x")

## 7 — Elevation-based colorization

A piecewise-linear color palette maps per-facet mean elevation to RGB, approximating the
visual appearance of the Cambrian uplands:

| Elevation band | Color          |
|----------------|----------------|
| 25–150 m       | Deep green (valley floors, improved pasture) |
| 150–350 m      | Green → olive (lower slopes, rough grazing) |
| 350–550 m      | Olive → brown (upland moorland) |
| 550–750 m      | Brown → red-brown (exposed peat and rock) |
| 750–885 m      | Grey-white (summit crags and scree) |

*(Real Sentinel-2 imagery can be substituted via `colorize.py` — see §4 for download instructions.)*

In [ ]:
# Elevation palette: (stop fraction, R, G, B)
_PALETTE_STOPS = np.array([0.00, 0.18, 0.40, 0.62, 0.82, 1.00])
_PALETTE_COLORS = np.array([
    [ 35, 110,  18],  # deep green   — valley floor
    [ 80, 145,  35],  # green        — lower slope
    [148, 128,  62],  # olive        — upland pasture
    [128,  85,  42],  # brown        — high moorland
    [158, 105,  65],  # red-brown    — exposed rock
    [205, 195, 185],  # grey-white   — summit crags
], dtype=np.float64)


def elevation_colorize(tile: TerrainTileData) -> TerrainTileData:
    """Return a copy of tile with per-facet RGB colors mapped from elevation."""
    # ENU Up component + centroid height = absolute elevation in metres.
    up = tile.vertices[:, 2].astype(np.float64) + tile.centroid_height_m
    f_elev = (
        up[tile.indices[:, 0]]
        + up[tile.indices[:, 1]]
        + up[tile.indices[:, 2]]
    ) / 3.0

    elev_min = float(f_elev.min())
    elev_range = float(f_elev.max() - elev_min)
    t = np.clip((f_elev - elev_min) / max(elev_range, 1.0), 0.0, 1.0)

    colors = np.zeros((len(f_elev), 3), dtype=np.float64)
    for seg in range(len(_PALETTE_STOPS) - 1):
        lo, hi = _PALETTE_STOPS[seg], _PALETTE_STOPS[seg + 1]
        mask = (t >= lo) & (t <= hi)
        if not mask.any():
            continue
        frac = (t[mask] - lo) / (hi - lo)
        colors[mask] = (
            _PALETTE_COLORS[seg] * (1.0 - frac[:, None])
            + _PALETTE_COLORS[seg + 1] * frac[:, None]
        )

    return dataclasses.replace(
        tile, colors=np.clip(colors, 0, 255).astype(np.uint8)
    )


print("Colorizing ...", end=" ", flush=True)
t0 = time.time()
tile_l3_c = elevation_colorize(tile_l3)
tile_l4_c = elevation_colorize(tile_l4)
tile_l5_c = elevation_colorize(tile_l5)
print(f"done in {time.time()-t0:.2f}s")

## 8 — Export to `.las_terrain`

All three LOD tiles are packed into a single `.las_terrain` binary file.  A round-trip
read is performed to confirm the archive is valid.

In [ ]:
LAS_PATH = _TMP_DIR / "cad_west.las_terrain"
export_las_terrain([tile_l3_c, tile_l4_c, tile_l5_c], LAS_PATH)
size_mb = LAS_PATH.stat().st_size / 1e6
print(f"Exported {LAS_PATH.name}  ({size_mb:.1f} MB)")

# Round-trip verify
rt_tiles = read_las_terrain(LAS_PATH)
print(f"Read back {len(rt_tiles)} tiles:")
for t in rt_tiles:
    print(f"  LOD {t.lod}: {len(t.vertices):,} vertices, {len(t.indices):,} triangles")

## 9 — PyVista 3D visualisation

Three LODs displayed side-by-side with linked cameras.  Illumination uses two lights:

- **Sun** — warm white, 35 ° elevation, 220 ° geographic azimuth (SSW afternoon sun, typical UK)
- **Sky fill** — cool blue, soft ambient from directly above (open-sky diffuse)

**Vertical exaggeration = 5×** makes the modest Welsh relief clearly visible at this scale.

Rotate with left-click drag, zoom with scroll, pan with right-click drag.

In [ ]:
import pyvista as pv

pv.set_jupyter_backend("trame")

VERT_SCALE = 5.0  # vertical exaggeration


def tile_to_pv_mesh(tile: TerrainTileData, vert_scale: float = 1.0) -> pv.PolyData:
    """Convert TerrainTileData to pv.PolyData with per-cell RGB colors.

    ENU axes: East = +X, North = +Y, Up = +Z (scaled by vert_scale).
    """
    verts = tile.vertices.astype(np.float64).copy()
    verts[:, 2] *= vert_scale

    faces_in = tile.indices.astype(np.int64)
    n_f = len(faces_in)
    pv_faces = np.empty(n_f * 4, dtype=np.int64)
    pv_faces[0::4] = 3
    pv_faces[1::4] = faces_in[:, 0]
    pv_faces[2::4] = faces_in[:, 1]
    pv_faces[3::4] = faces_in[:, 2]

    mesh = pv.PolyData(verts, pv_faces)
    mesh.cell_data["colors"] = tile.colors  # (F, 3) uint8 0-255
    return mesh


# --- Daylight sun: 35 deg elevation, 220 deg azimuth (SSW afternoon, UK) ---
# Geographic azimuth: 0 = North, 90 = East, 180 = South, 270 = West
# ENU coordinates: East = +x, North = +y, Up = +z
sun_el = math.radians(35)
sun_az = math.radians(220)
sun_e = math.sin(sun_az) * math.cos(sun_el)
sun_n = math.cos(sun_az) * math.cos(sun_el)
sun_u = math.sin(sun_el)
SUN_DIST = 5.0e6  # 5000 km — effectively directional
SUN_POS = (sun_e * SUN_DIST, sun_n * SUN_DIST, sun_u * SUN_DIST * VERT_SCALE)

print(f"Sun vector (ENU): ({sun_e:.3f}, {sun_n:.3f}, {sun_u:.3f})")
print(f"Vertical exaggeration: {VERT_SCALE}x")

In [ ]:
# Build PyVista meshes
lod_tiles = [tile_l3_c, tile_l4_c, tile_l5_c]
lod_labels = [
    f"LOD 3  |  300 m spacing  |  {nf3:,} faces",
    f"LOD 4  |  1 km spacing   |  {nf4:,} faces",
    f"LOD 5  |  3 km spacing   |  {nf5:,} faces",
]
lod_meshes = [tile_to_pv_mesh(t, VERT_SCALE) for t in lod_tiles]

# --- 1x3 interactive plotter ---
pl = pv.Plotter(
    shape=(1, 3),
    window_size=(1500, 560),
    notebook=True,
    border=True,
    border_color="grey",
)
pl.set_background("#d8e8f0")  # light sky blue

for col, (mesh, label) in enumerate(zip(lod_meshes, lod_labels)):
    pl.subplot(0, col)

    pl.add_mesh(
        mesh,
        scalars="colors",
        rgb=True,
        lighting=True,
        smooth_shading=False,
        show_scalar_bar=False,
    )

    # Warm sunlight
    pl.add_light(pv.Light(
        position=SUN_POS,
        focal_point=(0.0, 0.0, 0.0),
        light_type="scene light",
        intensity=1.0,
        color=[1.00, 0.97, 0.88],
    ))
    # Cool sky fill
    sky_pos = (0.0, 0.0, 5.0e6 * VERT_SCALE)
    pl.add_light(pv.Light(
        position=sky_pos,
        focal_point=(0.0, 0.0, 0.0),
        light_type="scene light",
        intensity=0.35,
        color=[0.68, 0.80, 1.00],
    ))

    pl.add_text(label, position="upper_left", font_size=9, color="black")

    # Camera: looking from SSW at a slight elevation (isometric-ish)
    half = 50_000.0  # half-extent in metres
    pl.camera.position = (-half * 0.5, -half * 1.3, half * 0.7 * VERT_SCALE)
    pl.camera.focal_point = (0.0, 0.0, dem_array.mean() * 0.5 * VERT_SCALE)
    pl.camera.up = (0.0, 0.0, 1.0)

# Link cameras so rotating one panel rotates all
pl.link_views()

pl.show()

## 10 — Individual LOD deep-dive

The cell below renders a single LOD in a larger window for detailed inspection.
Change `LOD_INDEX` to 0 (L3), 1 (L4), or 2 (L5).

In [ ]:
LOD_INDEX = 1  # 0=L3, 1=L4, 2=L5

tile_sel = lod_tiles[LOD_INDEX]
mesh_sel = lod_meshes[LOD_INDEX]
label_sel = lod_labels[LOD_INDEX]

pl2 = pv.Plotter(window_size=(900, 600), notebook=True)
pl2.set_background("#d8e8f0")
pl2.add_mesh(
    mesh_sel,
    scalars="colors",
    rgb=True,
    lighting=True,
    smooth_shading=False,
    show_scalar_bar=False,
)
pl2.add_light(pv.Light(
    position=SUN_POS,
    focal_point=(0.0, 0.0, 0.0),
    light_type="scene light",
    intensity=1.0,
    color=[1.00, 0.97, 0.88],
))
pl2.add_light(pv.Light(
    position=(0.0, 0.0, 5.0e6 * VERT_SCALE),
    focal_point=(0.0, 0.0, 0.0),
    light_type="scene light",
    intensity=0.35,
    color=[0.68, 0.80, 1.00],
))
pl2.add_text(label_sel, position="upper_left", font_size=10, color="black")
pl2.add_text(
    f"Cad West, Wales  |  100 km grid  |  {VERT_SCALE:.0f}x vert. exag.",
    position="lower_right",
    font_size=8,
    color="black",
)

half = 50_000.0
pl2.camera.position = (-half * 0.5, -half * 1.3, half * 0.7 * VERT_SCALE)
pl2.camera.focal_point = (0.0, 0.0, dem_array.mean() * 0.5 * VERT_SCALE)
pl2.camera.up = (0.0, 0.0, 1.0)

pl2.show()